# ABSA Full Pipeline — Fine-tune + Baselines + XAI
**All-in-one notebook** cho demo và báo cáo.

| Bước | Nội dung |
|------|----------|
| 1 | Setup: clone repo, cài thư viện, tìm dataset |
| 2 | Fine-tune Transformer (BERT/RoBERTa/…) |
| 3 | Train Baselines (LSTM / BiLSTM / RNN) |
| 4 | So sánh kết quả các mô hình |
| 5 | XAI: LIME, SHAP, Integrated Gradients, Attention, Gradient-based |
| 6 | Lưu toàn bộ kết quả ra `/kaggle/working/` |

## Bước 1 — Setup

In [ ]:
import subprocess, os, glob

REPO_URL = 'https://github.com/haiyen040602/xai-absa.git'
REPO_DIR = '/kaggle/working/xai-transformer'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo to', REPO_DIR)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Repo already present, pulled latest.')

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Cài thư viện
!pip install -q transformers lime shap captum seaborn scikit-learn

In [ ]:
# Tìm dataset
import glob

def find_dataset():
    # 1. Local (khi chạy offline)
    for local in ['Datasets/FINAL_CLEANED_CORRECTED_SHUFFLED_DATASET_NO_DUPLICATE.jsonl',
                  '../Datasets/FINAL_CLEANED_CORRECTED_SHUFFLED_DATASET_NO_DUPLICATE.jsonl']:
        if os.path.exists(local):
            return os.path.abspath(local)
    # 2. Kaggle Input
    for pattern in ['/kaggle/input/*/*.jsonl', '/kaggle/input/*/*/*.jsonl']:
        matches = glob.glob(pattern)
        if matches:
            return matches[0]
    return None

DATASET_PATH = find_dataset()
if DATASET_PATH is None:
    raise FileNotFoundError(
        'Khong tim thay dataset JSONL. '
        'Vui long attach dataset vao Kaggle Input hoac dung Add Data.'
    )
print('Dataset:', DATASET_PATH)

## Bước 2 — Fine-tune Transformer

> Thay `MODEL_NAME` để chọn mô hình khác: `roberta-base`, `distilbert-base-uncased`, `albert-base-v2`, `xlnet-base-cased`.

In [ ]:
# ===== CẤU HÌNH =====
# Dùng cho baseline (Bước 3) và fallback khi không chạy multi-model
MODEL_NAME   = 'bert-base-uncased'

# Bật chế độ test nhiều Transformer models cùng lúc
RUN_MULTI_MODELS = True
MODEL_CANDIDATES = [
    'bert-base-uncased',
    'bert-large-uncased',
    'distilbert-base-uncased',
    'albert-base-v1',
    'albert-large-v1',
    'roberta-base',
    'roberta-large',
    'xlnet-base-cased',
]
SELECT_METRIC = 'macro_f1'  # metric để chọn model tốt nhất cho XAI

# Tối ưu tài nguyên Kaggle
SKIP_TRAINED_MODELS = True     # nếu đã có metrics.json thì bỏ qua model đó
RETRY_ON_OOM = True            # tự retry khi out-of-memory
MIN_BATCH_SIZE = 1
MIN_MAX_LEN = 64

# Override theo từng model để giảm rủi ro OOM
MODEL_OVERRIDES = {
    'bert-large-uncased': {'batch_size': 2, 'max_length': 96, 'epochs': 2},
    'roberta-large': {'batch_size': 2, 'max_length': 96, 'epochs': 2},
    'xlnet-base-cased': {'batch_size': 2, 'max_length': 96, 'epochs': 2},
    'albert-large-v1': {'batch_size': 4, 'max_length': 96, 'epochs': 2},
    'roberta-base': {'batch_size': 4, 'max_length': 128, 'epochs': 3},
    'bert-base-uncased': {'batch_size': 8, 'max_length': 128, 'epochs': 3},
    'distilbert-base-uncased': {'batch_size': 8, 'max_length': 128, 'epochs': 3},
    'albert-base-v1': {'batch_size': 8, 'max_length': 128, 'epochs': 3},
}

EPOCHS       = 3
BATCH_SIZE   = 8
LR           = 1e-5
MAX_LEN      = 128
OUTPUT_DIR   = '/kaggle/working/absa_outputs'
SEED         = 42
# =====================

import gc
import json
import numpy as np
import pandas as pd
import torch

BEST_MODEL_NAME = None
BEST_MODEL_PATH = None


def get_metric(metrics, key: str, default=np.nan):
    def _search(obj):
        if isinstance(obj, dict):
            if key in obj and obj[key] is not None:
                return obj[key]
            for v in obj.values():
                found = _search(v)
                if found is not None:
                    return found
        elif isinstance(obj, list):
            for v in obj:
                found = _search(v)
                if found is not None:
                    return found
        return None

    value = _search(metrics)
    if value is None:
        return default
    try:
        val = float(value)
        return val if not np.isnan(val) else default
    except Exception:
        return default


def is_oom_text(log_text: str) -> bool:
    if not log_text:
        return False
    s = log_text.lower()
    return (
        'out of memory' in s
        or 'cuda error: out of memory' in s
        or 'cublas_status_alloc_failed' in s
        or 'cuda out of memory' in s
    )


def release_cuda_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if RUN_MULTI_MODELS:
    multi_root = os.path.join(OUTPUT_DIR, 'multi_models')
    os.makedirs(multi_root, exist_ok=True)

    rows = []
    print('Running multi-model transformer benchmark (resource-aware mode)...')

    for mname in MODEL_CANDIDATES:
        safe_name = mname.replace('/', '_')
        model_out_dir = os.path.join(multi_root, safe_name)
        metrics_path = os.path.join(model_out_dir, 'transformer', 'metrics.json')

        base_cfg = {
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'max_length': MAX_LEN,
            'learning_rate': LR,
        }
        base_cfg.update(MODEL_OVERRIDES.get(mname, {}))

        if SKIP_TRAINED_MODELS and os.path.exists(metrics_path):
            with open(metrics_path, 'r', encoding='utf-8') as f:
                metrics = json.load(f)

            rows.append({
                'model_name': mname,
                'run_output_dir': model_out_dir,
                'return_code': 0,
                'status': 'skipped_existing',
                'attempts': 0,
                'used_batch_size': base_cfg['batch_size'],
                'used_max_length': base_cfg['max_length'],
                'used_epochs': base_cfg['epochs'],
                'metrics_path': metrics_path,
                'accuracy': get_metric(metrics, 'accuracy'),
                'macro_precision': get_metric(metrics, 'macro_precision'),
                'macro_recall': get_metric(metrics, 'macro_recall'),
                'macro_f1': get_metric(metrics, 'macro_f1'),
            })
            print(f"[SKIP] {mname}: existing metrics found")
            continue

        attempt_cfg = dict(base_cfg)
        attempts = 0
        final_return_code = 1
        status = 'failed'

        while True:
            attempts += 1
            cmd = [
                'python3', 'kaggle/run_absa_repro.py',
                '--experiment', 'transformer',
                '--model_name', mname,
                '--epochs', str(attempt_cfg['epochs']),
                '--batch_size', str(attempt_cfg['batch_size']),
                '--learning_rate', str(attempt_cfg['learning_rate']),
                '--max_length', str(attempt_cfg['max_length']),
                '--output_dir', model_out_dir,
                '--seed', str(SEED),
                '--dataset_path', DATASET_PATH,
            ]

            print(f"\nRunning ({attempts}) {mname} | bs={attempt_cfg['batch_size']}, max_len={attempt_cfg['max_length']}, epochs={attempt_cfg['epochs']}")
            result = subprocess.run(cmd, capture_output=True, text=True)
            final_return_code = result.returncode

            combined_log = (result.stdout or '') + '\n' + (result.stderr or '')
            oom = is_oom_text(combined_log)

            if final_return_code == 0 and os.path.exists(metrics_path):
                status = 'ok'
                print(f"[OK] {mname}")
                break

            if RETRY_ON_OOM and oom and attempt_cfg['batch_size'] > MIN_BATCH_SIZE:
                prev_bs = attempt_cfg['batch_size']
                prev_len = attempt_cfg['max_length']
                attempt_cfg['batch_size'] = max(MIN_BATCH_SIZE, prev_bs // 2)
                if attempt_cfg['batch_size'] == prev_bs and prev_bs > MIN_BATCH_SIZE:
                    attempt_cfg['batch_size'] = prev_bs - 1
                attempt_cfg['max_length'] = max(MIN_MAX_LEN, prev_len - 16)
                status = 'retry_oom'
                print(f"[OOM] {mname}: retry with bs={attempt_cfg['batch_size']}, max_len={attempt_cfg['max_length']}")
                release_cuda_cache()
                continue

            if oom:
                status = 'failed_oom'
                print(f"[FAILED OOM] {mname}")
            else:
                status = 'failed_other'
                err_tail = (result.stderr or '').strip().splitlines()
                print(f"[FAILED] {mname}: return_code={final_return_code}")
                if err_tail:
                    print('stderr tail:', err_tail[-1])
            break

        metrics = {}
        if os.path.exists(metrics_path):
            with open(metrics_path, 'r', encoding='utf-8') as f:
                metrics = json.load(f)

        rows.append({
            'model_name': mname,
            'run_output_dir': model_out_dir,
            'return_code': final_return_code,
            'status': status,
            'attempts': attempts,
            'used_batch_size': attempt_cfg['batch_size'],
            'used_max_length': attempt_cfg['max_length'],
            'used_epochs': attempt_cfg['epochs'],
            'metrics_path': metrics_path,
            'accuracy': get_metric(metrics, 'accuracy'),
            'macro_precision': get_metric(metrics, 'macro_precision'),
            'macro_recall': get_metric(metrics, 'macro_recall'),
            'macro_f1': get_metric(metrics, 'macro_f1'),
        })

        # dọn VRAM giữa các model
        release_cuda_cache()

    multi_df = pd.DataFrame(rows)
    display(multi_df)

    csv_path = os.path.join(OUTPUT_DIR, 'multi_model_results.csv')
    multi_df.to_csv(csv_path, index=False)
    print('Saved:', csv_path)

    metric_candidates = [SELECT_METRIC, 'macro_f1', 'accuracy']
    metric = None
    for m in metric_candidates:
        if m in multi_df.columns and multi_df[m].notna().any():
            metric = m
            break

    success_df = multi_df[multi_df['return_code'] == 0].copy()
    if len(success_df) == 0:
        raise RuntimeError('Khong co model nao return_code=0 de chon cho XAI.')

    if metric is None:
        success_df = success_df.sort_values(by=['accuracy'], ascending=False, na_position='last')
        metric = 'accuracy'
        valid_df = success_df
    else:
        valid_df = success_df[success_df[metric].notna()].copy()
        if len(valid_df) == 0:
            valid_df = success_df
            metric = 'accuracy'
        sort_cols = [metric, 'accuracy'] if metric != 'accuracy' else ['accuracy']
        valid_df = valid_df.sort_values(by=sort_cols, ascending=False, na_position='last')
    best_row = valid_df.iloc[0]

    BEST_MODEL_NAME = best_row['model_name']
    BEST_MODEL_PATH = os.path.join(best_row['run_output_dir'], 'transformer')

    # Đồng bộ MODEL_NAME để các bước sau dùng model tốt nhất
    MODEL_NAME = BEST_MODEL_NAME

    print(f"\nBest model by {metric}: {BEST_MODEL_NAME}")
    print('BEST_MODEL_PATH:', BEST_MODEL_PATH)

else:
    cmd = [
        'python3', 'kaggle/run_absa_repro.py',
        '--experiment', 'transformer',
        '--model_name', MODEL_NAME,
        '--epochs', str(EPOCHS),
        '--batch_size', str(BATCH_SIZE),
        '--learning_rate', str(LR),
        '--max_length', str(MAX_LEN),
        '--output_dir', OUTPUT_DIR,
        '--seed', str(SEED),
        '--dataset_path', DATASET_PATH,
    ]

    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=False, text=True)
    print('Return code:', result.returncode)

    BEST_MODEL_NAME = MODEL_NAME
    BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, 'transformer')
    print('BEST_MODEL_PATH:', BEST_MODEL_PATH)


## Bước 3 — Train Baselines (LSTM / BiLSTM / RNN)

In [ ]:
# ===== CẤU HÌNH BASELINE =====
BL_EPOCHS   = 10
BL_BATCH    = 8
BL_LR       = 1e-3
BL_MAX_LEN  = 50
# ==============================

# Tiết kiệm tài nguyên: chỉ train baseline, không train lại transformer ở Bước 3
baseline_experiments = ['lstm', 'bilstm', 'rnn']
baseline_results = []

for exp_name in baseline_experiments:
    cmd_bl = [
        'python3', 'kaggle/run_absa_repro.py',
        '--experiment', exp_name,
        '--model_name', MODEL_NAME,
        '--epochs', str(EPOCHS),
        '--batch_size', str(BATCH_SIZE),
        '--learning_rate', str(LR),
        '--max_length', str(MAX_LEN),
        '--baseline_epochs', str(BL_EPOCHS),
        '--baseline_batch_size', str(BL_BATCH),
        '--baseline_learning_rate', str(BL_LR),
        '--baseline_max_length', str(BL_MAX_LEN),
        '--output_dir', OUTPUT_DIR,
        '--seed', str(SEED),
        '--dataset_path', DATASET_PATH,
    ]

    print('\nRunning:', ' '.join(cmd_bl))
    result_bl = subprocess.run(cmd_bl, capture_output=False, text=True)
    baseline_results.append({'experiment': exp_name, 'return_code': result_bl.returncode})
    print('Return code:', result_bl.returncode)

import pandas as pd
display(pd.DataFrame(baseline_results))

## Bước 4 — So sánh kết quả các mô hình

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import json


table_path = os.path.join(OUTPUT_DIR, 'comparison_table.csv')
rows = []

# 1) Transformer metrics: ưu tiên lấy đầy đủ từ multi_model_results.csv
multi_results_path = os.path.join(OUTPUT_DIR, 'multi_model_results.csv')
if os.path.exists(multi_results_path):
    mm_df = pd.read_csv(multi_results_path)
    # Giữ tất cả model đã chạy thành công hoặc skip do đã có metrics
    if 'status' in mm_df.columns:
        mm_df = mm_df[mm_df['status'].isin(['ok', 'skipped_existing'])].copy()

    for _, r in mm_df.iterrows():
        model_name = r.get('model_name', 'unknown')

        # multi_model_results.csv có thể không chứa cột loss -> đọc fallback từ metrics.json
        loss_val = r.get('loss', np.nan)
        if pd.isna(loss_val):
            mp = r.get('metrics_path', None)
            if isinstance(mp, str) and os.path.exists(mp):
                try:
                    with open(mp, 'r', encoding='utf-8') as f:
                        payload = json.load(f)
                    test = payload.get('test', payload)
                    loss_val = test.get('loss', np.nan)
                except Exception:
                    loss_val = np.nan

        rows.append({
            'experiment': f"transformer ({model_name})",
            'accuracy': r.get('accuracy'),
            'macro_precision': r.get('macro_precision'),
            'macro_recall': r.get('macro_recall'),
            'macro_f1': r.get('macro_f1'),
            'loss': loss_val,
        })
else:
    # Fallback nếu không chạy multi-model: lấy 1 transformer như trước
    transformer_candidates = [
        os.path.join(OUTPUT_DIR, 'transformer', 'metrics.json'),
    ]
    if 'BEST_MODEL_PATH' in globals() and BEST_MODEL_PATH:
        transformer_candidates.append(os.path.join(BEST_MODEL_PATH, 'metrics.json'))

    transformer_metrics_path = None
    for p in transformer_candidates:
        if p and os.path.exists(p):
            transformer_metrics_path = p
            break

    if transformer_metrics_path:
        with open(transformer_metrics_path, 'r', encoding='utf-8') as f:
            payload = json.load(f)
        test = payload.get('test', payload)
        model_tag = payload.get('model_name', 'transformer')
        rows.append({
            'experiment': f"transformer ({model_tag})",
            'accuracy': test.get('accuracy'),
            'macro_precision': test.get('macro_precision'),
            'macro_recall': test.get('macro_recall'),
            'macro_f1': test.get('macro_f1'),
            'loss': test.get('loss'),
        })

# 2) Baseline metrics
for exp_name in ['lstm', 'bilstm', 'rnn']:
    metrics_path = os.path.join(OUTPUT_DIR, exp_name, 'metrics.json')
    if not os.path.exists(metrics_path):
        continue
    with open(metrics_path, 'r', encoding='utf-8') as f:
        payload = json.load(f)
    test = payload.get('test', payload)
    rows.append({
        'experiment': exp_name,
        'accuracy': test.get('accuracy'),
        'macro_precision': test.get('macro_precision'),
        'macro_recall': test.get('macro_recall'),
        'macro_f1': test.get('macro_f1'),
        'loss': test.get('loss'),
    })

if rows:
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=['experiment'], keep='last')
    df = df.sort_values(by='experiment').reset_index(drop=True)
    df.to_csv(table_path, index=False)
    print('Rebuilt and saved:', table_path)
else:
    if not os.path.exists(table_path):
        raise FileNotFoundError(
            'Khong tim thay metrics cho transformer/lstm/bilstm/rnn. '
            'Hay chay lai buoc 2 va buoc 3 truoc.'
        )
    df = pd.read_csv(table_path)

# Display
display(df)

# Plot all metrics
plot_df = df.copy()
metric_cols = ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1']
for c in metric_cols + ['loss']:
    plot_df[c] = pd.to_numeric(plot_df[c], errors='coerce')

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(15, 10), gridspec_kw={'height_ratios': [2.3, 1.2]}
)

x = np.arange(len(plot_df))
width = 0.18
metric_colors = {
    'accuracy': '#4C78A8',
    'macro_precision': '#54A24B',
    'macro_recall': '#E45756',
    'macro_f1': '#F58518',
}

for i, m in enumerate(metric_cols):
    vals = plot_df[m].values
    bars = ax1.bar(x + (i - 1.5) * width, vals, width, label=m, color=metric_colors[m])
    for b in bars:
        h = b.get_height()
        if pd.notna(h):
            ax1.text(
                b.get_x() + b.get_width() / 2,
                h + 0.008,
                f'{h:.3f}',
                ha='center',
                va='bottom',
                fontsize=7,
            )

ax1.set_ylabel('Score')
ax1.set_ylim(0, 1.10)
ax1.set_title('Model Comparison — Accuracy / Precision / Recall / Macro F1')
ax1.set_xticks(x)
ax1.set_xticklabels(plot_df['experiment'], rotation=25, ha='right')
ax1.legend(ncol=2)

loss_vals = plot_df['loss'].values
bars_loss = ax2.bar(x, loss_vals, color='#7A7A7A', width=0.6, label='loss')
for b in bars_loss:
    h = b.get_height()
    if pd.notna(h):
        ax2.text(
            b.get_x() + b.get_width() / 2,
            h + (0.01 if h < 1 else 0.02),
            f'{h:.3f}',
            ha='center',
            va='bottom',
            fontsize=7,
        )

ax2.set_ylabel('Loss')
ax2.set_title('Model Comparison — Loss')
ax2.set_xticks(x)
ax2.set_xticklabels(plot_df['experiment'], rotation=25, ha='right')
ax2.legend()

plt.tight_layout()
chart_path = os.path.join(OUTPUT_DIR, 'comparison_chart.png')
plt.savefig(chart_path, dpi=150)
plt.show()
print('Saved:', chart_path)


In [ ]:
# Confusion matrix subplots (Transformer best + LSTM + BiLSTM + RNN)
import matplotlib.image as mpimg
import pandas as pd

# Resolve best transformer confusion matrix path
best_transformer_name = 'transformer'
transformer_cm_candidates = [
    os.path.join(OUTPUT_DIR, 'transformer', 'confusion_matrix.png'),
]

if 'BEST_MODEL_PATH' in globals() and BEST_MODEL_PATH:
    transformer_cm_candidates.insert(0, os.path.join(BEST_MODEL_PATH, 'confusion_matrix.png'))
    if 'BEST_MODEL_NAME' in globals() and BEST_MODEL_NAME:
        best_transformer_name = str(BEST_MODEL_NAME)

# Fallback: infer best from multi_model_results.csv if needed
multi_results_path = os.path.join(OUTPUT_DIR, 'multi_model_results.csv')
if os.path.exists(multi_results_path):
    mm_df = pd.read_csv(multi_results_path)
    if 'return_code' in mm_df.columns:
        mm_df = mm_df[mm_df['return_code'] == 0].copy()
    if len(mm_df) > 0:
        metric_candidates = [
            SELECT_METRIC if 'SELECT_METRIC' in globals() else 'macro_f1',
            'macro_f1',
            'accuracy',
        ]
        chosen_metric = None
        for m in metric_candidates:
            if m in mm_df.columns and mm_df[m].notna().any():
                chosen_metric = m
                break
        if chosen_metric is None:
            chosen_metric = 'accuracy'
        mm_df = mm_df.sort_values(by=[chosen_metric], ascending=False, na_position='last')
        best_row = mm_df.iloc[0]
        best_transformer_name = str(best_row.get('model_name', best_transformer_name))
        run_out = best_row.get('run_output_dir', None)
        if isinstance(run_out, str):
            transformer_cm_candidates.insert(0, os.path.join(run_out, 'transformer', 'confusion_matrix.png'))

transformer_cm_path = None
for p in transformer_cm_candidates:
    if p and os.path.exists(p):
        transformer_cm_path = p
        break

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: best transformer
ax0 = axes.flat[0]
if transformer_cm_path and os.path.exists(transformer_cm_path):
    img = mpimg.imread(transformer_cm_path)
    ax0.imshow(img)
    ax0.set_title(f'TRANSFORMER ({best_transformer_name})', fontsize=12)
else:
    miss = transformer_cm_candidates[0] if transformer_cm_candidates else os.path.join(OUTPUT_DIR, 'transformer', 'confusion_matrix.png')
    ax0.text(0.5, 0.5, 'Missing file:\n' + miss, ha='center', va='center', fontsize=9)
    ax0.set_title('TRANSFORMER (not found)', fontsize=12)
ax0.axis('off')

# Panels 2-4: baselines
for ax, mname in zip(axes.flat[1:], ['lstm', 'bilstm', 'rnn']):
    img_path = os.path.join(OUTPUT_DIR, mname, 'confusion_matrix.png')
    if os.path.exists(img_path):
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.set_title(mname.upper(), fontsize=14)
    else:
        ax.text(0.5, 0.5, 'Missing file:\n' + img_path, ha='center', va='center', fontsize=9)
        ax.set_title(mname.upper() + ' (not found)', fontsize=12)
    ax.axis('off')

plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')
plt.tight_layout()
cm_grid_path = os.path.join(OUTPUT_DIR, 'confusion_matrix_grid.png')
plt.savefig(cm_grid_path, dpi=150)
plt.show()
print('Saved:', cm_grid_path)


## Bước 5 — XAI Explanations

Load model fine-tuned tốt nhất, rồi chạy 5 kỹ thuật giải thích.

> Thay `EXPLAIN_TEXT` và `EXPLAIN_ASPECT` để thử với câu và khía cạnh khác.

In [ ]:
import torch, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
import shap
from captum.attr import LayerIntegratedGradients

sns.set_theme(style='whitegrid')

# ===== CẤU HÌNH XAI =====
EXPLAIN_TEXT   = 'The food was amazing but the service was very slow.'
EXPLAIN_ASPECT = 'service'
XAI_OUT_DIR    = '/kaggle/working/xai_outputs'
os.makedirs(XAI_OUT_DIR, exist_ok=True)
# =========================

# Ưu tiên model tốt nhất từ Bước 2 (nếu có)
MODEL_PATH = None
if 'BEST_MODEL_PATH' in globals() and BEST_MODEL_PATH and os.path.exists(os.path.join(BEST_MODEL_PATH, 'config.json')):
    MODEL_PATH = BEST_MODEL_PATH

# Fallback: tìm model fine-tuned trong output/input
if MODEL_PATH is None:
    for p in [
        os.path.join(OUTPUT_DIR, 'transformer'),
        *glob.glob('/kaggle/input/*/transformer'),
        *glob.glob('/kaggle/input/*/*transformer*'),
    ]:
        if os.path.exists(os.path.join(p, 'config.json')):
            MODEL_PATH = p
            break

if MODEL_PATH is None:
    raise FileNotFoundError(
        'Khong tim thay model fine-tuned. '
        'Hay chay Buoc 2 truoc hoac attach model vao Kaggle Input.'
    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

name_hint = (model.config._name_or_path or '').lower()
if 'roberta' in name_hint:
    SEP = '</s></s>'
elif 'xlnet' in name_hint:
    SEP = '<sep>'
else:
    SEP = '[SEP]'

label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
print('Model loaded from:', MODEL_PATH)
if 'BEST_MODEL_NAME' in globals() and BEST_MODEL_NAME:
    print('Best model selected from Step 2:', BEST_MODEL_NAME)
print('Separator:', SEP, '| Device:', device)

def build_input(t, a): return t + ' ' + SEP + ' ' + a

def predict_proba(texts, aspects=None):
    asp = aspects if aspects else [EXPLAIN_ASPECT] * len(texts)
    probs = []
    for t, a in zip(texts, asp):
        enc = tokenizer(build_input(t, a), return_tensors='pt',
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model(**enc).logits
            probs.append(torch.softmax(out, dim=-1).cpu().numpy()[0])
    return np.array(probs)

probs = predict_proba([EXPLAIN_TEXT])[0]
pred = int(np.argmax(probs))
print('\nInput:', build_input(EXPLAIN_TEXT, EXPLAIN_ASPECT))
print('Predicted:', label_map[pred], '| Probs:', np.round(probs, 4))

### 5.1 — LIME

In [ ]:
from collections import defaultdict

# Format theo LIME.py mẫu: giữ explain_instance + in trọng số theo từng class
explainer = LimeTextExplainer(class_names=['negative', 'neutral', 'positive'])

text = EXPLAIN_TEXT
aspect = EXPLAIN_ASPECT


def predict_sentiment(texts, aspects):
    predictions = []
    for t, a in zip(texts, aspects):
        probs = predict_proba([t], [a])[0]
        predictions.append(probs)
    return np.array(predictions)


def lime_predict(texts):
    aspects = [aspect] * len(texts)
    return predict_sentiment(texts, aspects)


def explain_instance_with_class_columns(explainer, text, predict_fn, num_features, num_samples):
    print(f'Explaining instance with text: {text}')
    explanation = explainer.explain_instance(
        text,
        predict_fn,
        num_features=num_features,
        num_samples=num_samples,
        top_labels=len(explainer.class_names),
    )
    print('Explanation generated.')
    print(f'Class names: {explanation.class_names}')
    print(f'Local explanation: {explanation.local_exp}')

    class_weights = defaultdict(list)

    for class_index, class_name in enumerate(explainer.class_names):
        if class_index not in explanation.local_exp:
            print(f'Class index {class_index} not found in local_exp')
            continue

        print(f"\nProcessing class '{class_name}' with index {class_index}")
        for feature, weight in explanation.local_exp[class_index]:
            word = explanation.domain_mapper.indexed_string.word(feature)
            print(f'Feature index: {feature}, Word: {word}, Weight: {weight}')
            class_weights[class_name].append((word, weight))

    for class_name, weights in class_weights.items():
        print(f"\nWeights for class '{class_name}':")
        for word, weight in weights:
            print(f'{word}: {weight}')

    return explanation


exp = explain_instance_with_class_columns(
    explainer,
    text,
    lime_predict,
    num_features=5,
    num_samples=1000,
)

# Hiển thị đúng kiểu notebook view của LIME
exp.show_in_notebook(text=True)


### 5.2 — SHAP

In [ ]:
import shap

# Format theo SHAP.py mẫu: text_plot là visualization chính
# Dùng text/aspect từ cấu hình notebook
text = EXPLAIN_TEXT
aspect = EXPLAIN_ASPECT


def predict_sentiment(texts):
    inputs = tokenizer(texts, return_tensors='pt', truncation=True, padding=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.softmax(outputs.logits, dim=-1)
    return predictions.detach().cpu().numpy()


# SHAP explainer
explainer = shap.Explainer(
    lambda x: predict_sentiment([f"{t} {SEP} {aspect}" for t in x]),
    tokenizer,
)
# Có thể thử các separator khác nếu cần:
# explainer = shap.Explainer(lambda x: predict_sentiment([f"{t} [SEP] {aspect}" for t in x]), tokenizer)
# explainer = shap.Explainer(lambda x: predict_sentiment([f"{t} <sep> {aspect}" for t in x]), tokenizer)

# Prepare input text
input_text = [text]

# Compute SHAP values
shap_values = explainer(input_text)

# Visualization
shap.initjs()
shap.text_plot(shap_values)


### 5.3 — Integrated Gradients (Captum)

In [ ]:
from captum.attr import LayerIntegratedGradients, visualization
import torch

# Format theo INTEGRATED_GRADIENTS.py mẫu
texts = [
    EXPLAIN_TEXT,
    EXPLAIN_TEXT,
    EXPLAIN_TEXT,
]
aspects = [
    'food',
    'service',
    EXPLAIN_ASPECT,
]

# True labels demo (0: negative, 1: neutral, 2: positive)
true_labels = [2, 0, 1]

# Nếu notebook đã có label_map thì dùng lại, nếu chưa thì tạo mới
if 'label_map' not in globals():
    label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

# Initialize LayerIntegratedGradients
lig = LayerIntegratedGradients(
    lambda input_ids, attention_mask: model(input_ids=input_ids, attention_mask=attention_mask).logits,
    model.base_model.embeddings,
)


def construct_input_ref_pair(text, aspect, tokenizer):
    input_text = f"{text} {SEP} {aspect}"
    # input_text = f"{text} </s></s> {aspect}"
    # input_text = f"{text} [SEP] {aspect}"
    # input_text = f"{text} <sep> {aspect}"
    tokenized_input = tokenizer(input_text, padding=True, truncation=True, return_tensors='pt')
    input_ids = tokenized_input['input_ids'].to(device)
    attention_mask = tokenized_input['attention_mask'].to(device)
    return input_ids, attention_mask


all_case_scores = []

for idx, (text, aspect, true_label) in enumerate(zip(texts, aspects, true_labels), start=1):
    # Construct input
    input_ids, attention_mask = construct_input_ref_pair(text, aspect, tokenizer)

    # Predict sentiment
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predicted_class = torch.argmax(logits, dim=1).item()

    # Compute attributions
    target = predicted_class
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=None,
        target=target,
        additional_forward_args=(attention_mask,),
        return_convergence_delta=True,
    )

    # Summarize attributions
    attr_sum = attributions.sum(dim=-1)
    norm = torch.norm(attr_sum)
    if norm.item() == 0:
        attributions_sum = attr_sum.squeeze(0)
    else:
        attributions_sum = (attr_sum / norm).squeeze(0)

    # Convert input IDs to tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0).detach().cpu())

    # Print token scores
    print(f"\n=== Case {idx} ===")
    print(f"Text: {text}")
    print(f"Aspect: {aspect}")
    print(f"Predicted Sentiment: {label_map[predicted_class]}")
    print(f"True Sentiment: {label_map.get(true_label, 'unknown')}")
    print('Word Importance Scores:')
    for token, score in zip(tokens, attributions_sum.detach().cpu().tolist()):
        print(f"{token}: {score:.4f}")

    # Visualize attributions (Captum notebook format)
    visualization.visualize_text([
        visualization.VisualizationDataRecord(
            attributions_sum.detach().cpu(),
            torch.max(torch.softmax(logits, dim=1)).item(),
            label_map[predicted_class],
            label_map.get(true_label, 'unknown'),
            label_map[predicted_class],
            attributions_sum.sum().item(),
            tokens,
            delta.detach().cpu(),
        )
    ])

    all_case_scores.append({
        'case': idx,
        'text': text,
        'aspect': aspect,
        'predicted': label_map[predicted_class],
        'true': label_map.get(true_label, 'unknown'),
        'delta': float(delta.mean().item()) if hasattr(delta, 'mean') else float(delta),
    })

# Optional save: bar chart cho case cuối để có PNG tải về
last_scores = attributions_sum.detach().cpu().numpy()
colors_ig = ['#d62728' if s > 0 else '#1f77b4' for s in last_scores]
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(tokens)), last_scores, color=colors_ig)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(list(range(len(tokens))))
ax.set_xticklabels(tokens, rotation=90)
ax.set_ylabel('Normalized attribution')
ax.set_title('Integrated Gradients (last case)')
plt.tight_layout()
ig_path = os.path.join(XAI_OUT_DIR, 'integrated_gradients.png')
plt.savefig(ig_path, dpi=150)
plt.show()
print('Saved:', ig_path)


### 5.4 — Attention Weights (last layer, mean over heads)

In [ ]:
# Format theo ATTENTION_WEIGHTS.py mẫu
att_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    output_attentions=True,
).to(device)
att_model.eval()

# Define input text and aspect
text = EXPLAIN_TEXT
aspect = EXPLAIN_ASPECT

input_text = f"{text} {SEP} {aspect}"
# input_text = f"{text} <sep> {aspect}"
# input_text = f"{text} [SEP] {aspect}"

# Tokenize input text
tokens = tokenizer.tokenize(input_text)
token_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)

# Get model output
with torch.no_grad():
    outputs = att_model(token_ids)

# Extract attention weights from the last layer
attention_weights = outputs.attentions[-1]

# Average attention weights across all attention heads
avg_attention_weights = torch.mean(attention_weights, dim=1).squeeze(0).detach().cpu().numpy()

# Convert token IDs to words
word_tokens = tokenizer.convert_ids_to_tokens(token_ids.squeeze().detach().cpu().tolist())

# Visualize attention weights
plt.figure(figsize=(12, 10))
plt.imshow(avg_attention_weights, cmap='coolwarm', interpolation='nearest')

plt.xticks(ticks=range(len(word_tokens)), labels=word_tokens, rotation=45)
plt.yticks(ticks=range(len(word_tokens)), labels=word_tokens)

# Annotate heatmap with attention weights
for i in range(len(word_tokens)):
    for j in range(len(word_tokens)):
        plt.text(j, i, f'{avg_attention_weights[i, j]:.2f}', ha='center', va='center', color='white', fontsize=6)

plt.xlabel('Source Tokens')
plt.ylabel('Target Tokens')
plt.title(f'Attention Weights from the Last Layer (Aspect: {aspect})')
plt.colorbar(label='Attention Weight')
plt.tight_layout()

att_path = os.path.join(XAI_OUT_DIR, 'attention_heatmap.png')
plt.savefig(att_path, dpi=150)
plt.show()
print('Saved:', att_path)


### 5.5 — Grad-CAM


In [ ]:
# Format theo GRAD-CAM.py mẫu
import seaborn as sns


class GradCam:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks
        self.target_layer.register_forward_hook(self.save_activations)
        self.target_layer.register_backward_hook(self.save_gradients)

    def save_activations(self, module, input, output):
        # Handle tuple output (e.g., some model internals)
        if isinstance(output, tuple):
            self.activations = output[0]
        else:
            self.activations = output

    def save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def compute_gradcam(self, input_text, aspect, tokenizer):
        # Combine input text and aspect
        combined_input = f"{input_text} {SEP} {aspect}"
        # combined_input = f"{input_text} [SEP] {aspect}"
        # combined_input = f"{input_text} <sep> {aspect}"

        # Tokenize the combined input
        inputs = tokenizer(combined_input, return_tensors='pt').to(device)

        self.model.eval()

        # Forward pass
        outputs = self.model(**inputs)

        # Get target class and score
        target_class = torch.argmax(outputs.logits, dim=1).item()
        target_score = outputs.logits[0, target_class]

        # Backward pass
        self.model.zero_grad()
        target_score.backward()

        # Extract gradients and activations
        gradients = self.gradients.mean(dim=1).squeeze()
        activations = self.activations.squeeze()

        # Compute relevance per token
        weights = gradients
        cam = torch.zeros(activations.size(0), device=activations.device)
        for i in range(activations.size(0)):
            cam[i] = (weights * activations[i]).sum()

        # ReLU + normalize
        cam = torch.relu(cam)
        if cam.max() > 0:
            cam = cam / cam.max()

        return cam, inputs['input_ids'][0], target_class


# Example from notebook configuration
input_text = EXPLAIN_TEXT
aspect = EXPLAIN_ASPECT

# Select target layer by model type
name_lower = (model.config._name_or_path or MODEL_PATH).lower()
if 'roberta' in name_lower:
    target_layer = model.roberta.encoder.layer[-1].output
elif 'albert' in name_lower:
    target_layer = model.albert.encoder.albert_layer_groups[-1].albert_layers[-1].ffn_output
elif 'distilbert' in name_lower:
    target_layer = model.distilbert.transformer.layer[-1].ffn.lin2
elif 'xlnet' in name_lower:
    target_layer = model.transformer.layer[-1]
else:
    target_layer = model.bert.encoder.layer[-1].output

# Initialize Grad-CAM
gradcam = GradCam(model, target_layer)

# Compute Grad-CAM scores
cam_scores, input_ids, target_class = gradcam.compute_gradcam(input_text, aspect, tokenizer)

# Convert token IDs to tokens
tokens = tokenizer.convert_ids_to_tokens(input_ids.detach().cpu().tolist())

# Combine tokens with scores
highlighted_text = [(token, score.item()) for token, score in zip(tokens, cam_scores)]

# Print token relevance scores
print(f"Grad-CAM Relevance Scores for Aspect '{aspect}' in Input Text:")
for token, score in highlighted_text:
    print(f"{token}: {score:.4f}")

# Predicted polarity
polarity_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
predicted_polarity = polarity_mapping.get(target_class, 'unknown')
print(f"\nPredicted Polarity for Aspect '{aspect}': {predicted_polarity}")

# Visualize heatmap (no cell annotations)
plt.figure(figsize=(12, 6))
ax = sns.heatmap([cam_scores.detach().cpu().numpy()], cmap='coolwarm', cbar_kws={'label': 'Relevance Score'})
ax.set_title(f'Grad-CAM Relevance Scores for Aspect "{aspect}" in Input Text')
ax.set_xlabel('Tokens')
ax.set_xticks(np.arange(len(tokens)) + 0.5)
ax.set_xticklabels(tokens, rotation=90)
plt.yticks([], [])
plt.tight_layout()

grad_path = os.path.join(XAI_OUT_DIR, 'gradcam.png')
plt.savefig(grad_path, dpi=150)
plt.show()
print('Saved:', grad_path)


## Bước 6 — Tóm tắt kết quả

In [ ]:
import json

summary = {
    'model_used': MODEL_PATH,
    'separator': SEP,
    'device': str(device),
    'explain_text': EXPLAIN_TEXT,
    'explain_aspect': EXPLAIN_ASPECT,
    'predicted_label': label_map[pred],
    'probabilities': {label_map[i]: float(probs[i]) for i in range(3)},
    'xai_outputs': {
        'lime': lime_path,
        'integrated_gradients': ig_path,
        'attention_heatmap': att_path,
        'gradient_token_importance': grad_path,
    },
}

summary_path = os.path.join(XAI_OUT_DIR, 'xai_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('=== XAI Summary ===')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\nAll outputs saved to:', XAI_OUT_DIR)

In [ ]:
# Bước 7 — Gom artifacts để tải về từ Kaggle
import os
import glob
import json
import shutil
from datetime import datetime

EXPORT_ROOT = '/kaggle/working'
EXPORT_NAME = 'absa_export_bundle'
EXPORT_DIR = os.path.join(EXPORT_ROOT, EXPORT_NAME)

os.makedirs(EXPORT_DIR, exist_ok=True)

files_to_copy = []
dirs_to_copy = []

# 1) Core outputs
if 'OUTPUT_DIR' in globals() and OUTPUT_DIR and os.path.exists(OUTPUT_DIR):
    dirs_to_copy.append((OUTPUT_DIR, 'absa_outputs'))

if 'XAI_OUT_DIR' in globals() and XAI_OUT_DIR and os.path.exists(XAI_OUT_DIR):
    dirs_to_copy.append((XAI_OUT_DIR, 'xai_outputs'))

# 2) Best model folder
if 'BEST_MODEL_PATH' in globals() and BEST_MODEL_PATH and os.path.exists(BEST_MODEL_PATH):
    dirs_to_copy.append((BEST_MODEL_PATH, 'best_model'))

# 3) Important single files (if present)
candidate_files = [
    '/kaggle/working/absa_outputs/comparison_table.csv',
    '/kaggle/working/absa_outputs/comparison_chart.png',
    '/kaggle/working/absa_outputs/confusion_matrix_grid.png',
    '/kaggle/working/absa_outputs/multi_model_results.csv',
    '/kaggle/working/xai_outputs/xai_summary.json',
    '/kaggle/working/xai_outputs/lime_explanation.png',
    '/kaggle/working/xai_outputs/shap_explanation.png',
    '/kaggle/working/xai_outputs/integrated_gradients.png',
    '/kaggle/working/xai_outputs/attention_heatmap.png',
    '/kaggle/working/xai_outputs/gradcam.png',
]

for fp in candidate_files:
    if os.path.exists(fp):
        files_to_copy.append(fp)

# 4) Extra image sweep for convenience
for pattern in [
    '/kaggle/working/absa_outputs/**/*.png',
    '/kaggle/working/xai_outputs/**/*.png',
    '/kaggle/working/absa_outputs/**/*.jpg',
    '/kaggle/working/xai_outputs/**/*.jpg',
]:
    for fp in glob.glob(pattern, recursive=True):
        if os.path.isfile(fp):
            files_to_copy.append(fp)

files_to_copy = sorted(set(files_to_copy))

# Copy directories
for src_dir, dst_name in dirs_to_copy:
    dst_dir = os.path.join(EXPORT_DIR, dst_name)
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)

# Copy standalone files (flatten under "files")
files_dir = os.path.join(EXPORT_DIR, 'files')
os.makedirs(files_dir, exist_ok=True)
for src_fp in files_to_copy:
    dst_fp = os.path.join(files_dir, os.path.basename(src_fp))
    if os.path.exists(dst_fp):
        base, ext = os.path.splitext(os.path.basename(src_fp))
        idx = 2
        while True:
            cand = os.path.join(files_dir, f'{base}_{idx}{ext}')
            if not os.path.exists(cand):
                dst_fp = cand
                break
            idx += 1
    shutil.copy2(src_fp, dst_fp)

# Write manifest for quick check
manifest = {
    'created_at': datetime.utcnow().isoformat() + 'Z',
    'best_model_name': globals().get('BEST_MODEL_NAME', None),
    'best_model_path': globals().get('BEST_MODEL_PATH', None),
    'model_path_used_for_xai': globals().get('MODEL_PATH', None),
    'copied_dirs': [dst_name for _, dst_name in dirs_to_copy],
    'num_files_flattened': len(files_to_copy),
}

manifest_path = os.path.join(EXPORT_DIR, 'manifest.json')
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

# Zip for one-click download in Kaggle output
zip_base = os.path.join(EXPORT_ROOT, EXPORT_NAME)
zip_path = shutil.make_archive(zip_base, 'zip', EXPORT_DIR)

print('Export folder:', EXPORT_DIR)
print('Export zip   :', zip_path)
print('Manifest     :', manifest_path)
print('Done. Open the Files panel on the right in Kaggle and download the .zip file.')